# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess
import pandas as pd

# --- Bootstrap: locate the repo root so this runs from anywhere (local or Colab) ---
CSV = "data/raw/content_refresh_anonymized.csv"
REPO_URL = "https://github.com/thany-8/content-refresh-prioritizer"
REPO_DIR = "content-refresh-prioritizer"

def find_root(start, marker=CSV, up=6):
    """Walk up from `start` until a dir containing `marker` is found (repo root)."""
    d = os.path.abspath(start)
    for _ in range(up + 1):
        if os.path.exists(os.path.join(d, marker)):
            return d
        d = os.path.dirname(d)
    return None

root = find_root(os.getcwd())
if root is None:  # e.g. a fresh Colab VM: clone the public repo, then use it
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    root = os.path.abspath(REPO_DIR)
os.chdir(root)

assert os.path.exists(CSV), f"starter CSV not found under {root}"
df = pd.read_csv(CSV)
print("loaded", df.shape[0], "rows x", df.shape[1], "cols from", CSV)

loaded 30000 rows x 44 cols from data/raw/content_refresh_anonymized.csv


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one pseudonymized content item (a page), for one client.** `content_id` is unique across the 30,000 rows (32 clients), and `(client_id, content_id)` never repeats.

**Time window:** every metric is a single **trailing-90-day snapshot** ending at export — there is *no calendar date column*, so this is a cross-section, not a time series. Two nested windows live inside it: the 90-day totals (`*_90d`) and the 30-day comparison pair `*_last_30d` (days 0–30) vs `*_prev_30d` (days 31–60) that the trend is built from.

In [ ]:
# GRAIN + WINDOW  (pandas on the single-snapshot starter CSV is our query engine)
print("rows x cols:", df.shape)
print("clients:", df["client_id"].nunique())
print("content_id unique (one row per page):", df["content_id"].is_unique)
print("duplicate (client_id, content_id) rows:", int(df.duplicated(["client_id", "content_id"]).sum()))

# Time window: a single trailing-90-day snapshot per page (no calendar date column).
date_cols = [c for c in df.columns if "date" in c.lower()]
print("calendar date columns:", date_cols or "none -> one snapshot, not a time series")
print("90d coverage: days_with_impressions range =",
      int(df["days_with_impressions"].min()), "-", int(df["days_with_impressions"].max()))
print("30d compare windows: *_last_30d (days 0-30) vs *_prev_30d (days 31-60)")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every one of the 44 columns goes in **exactly one** bucket (the code cell enforces the split is complete and non-overlapping).

**Context** — grouping / joining / splitting only, never learned from

| Field | Why |
|---|---|
| `content_id` | pseudonymous page id — joins / grouping |
| `client_id` | pseudonymous client id — **grouped (client-holdout) splits** |

**Label / proxy** — the target and everything it is computed from, never a feature

| Field | Why |
|---|---|
| `trend_direction` | label source: `is_declining_label = (trend_direction == "down")` |
| `trend_pct` | defines `trend_direction` |
| `impressions_last_30d`, `impressions_prev_30d` | `trend_pct = (last30 − prev30) / prev30` — they **reconstruct** the label |

**Excluded** — each with a one-line why

| Field | Why excluded |
|---|---|
| `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d` | live in the label's recent-30d window and track the impressions trend → leakage risk |
| `provider_used`, `model_used` | how the article was generated (pipeline metadata, ~71% / ~19% missing) — not a content signal |

**Feature** — the remaining **32** columns: keyword context (`search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`), content properties (`word_count`, `char_count`, `content_age_days`, `days_since_last_update`, and the `*_tier` buckets), and whole-window 90-day behaviour + rates (`impressions_90d`…`scroll_events_90d`, `days_with_*`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`).

In [ ]:
# Every one of the 44 columns goes in exactly one bucket.
CONTEXT  = ["content_id", "client_id"]
LABEL    = ["trend_direction", "trend_pct",                 # target + what defines it
            "impressions_last_30d", "impressions_prev_30d"] # the trend's raw inputs
EXCLUDED = ["provider_used", "model_used",                  # generation metadata, not a feature
            "clicks_last_30d", "sessions_last_30d",
            "clicks_prev_30d", "sessions_prev_30d"]         # label-window 30d proxies -> leakage risk
FEATURE  = [c for c in df.columns if c not in CONTEXT + LABEL + EXCLUDED]

# Verify the partition is complete and disjoint (no column unclassified or double-counted).
buckets = {"FEATURE": FEATURE, "LABEL": LABEL, "CONTEXT": CONTEXT, "EXCLUDED": EXCLUDED}
allcols = [c for cols in buckets.values() for c in cols]
assert sorted(allcols) == sorted(df.columns), "every column must be classified exactly once"
assert len(allcols) == len(set(allcols)) == df.shape[1]
print("all", df.shape[1], "columns classified exactly once\n")
for name, cols in buckets.items():
    print(f"{name:8} {len(cols):2d}")
print("\nLABEL   :", LABEL)
print("EXCLUDED:", EXCLUDED)
print("CONTEXT :", CONTEXT)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The checks below confirm each claim: the **grain** holds (0 `content_id` appears twice), the **panel is uneven** even in this slice (rows/client vary widely), **missingness is patterned by `content_type`** (feedly articles have 100% missing keyword data; keyword articles ~28% missing `word_count`) — so a blind `fillna(0)` would smuggle content-type into the features — and the label input `trend_pct` is blank *exactly* when its denominator `impressions_prev_30d` is 0.

In [ ]:
# 1) GRAIN: group by the unit; any group > 1 breaks it.
print("grain check — content_ids appearing >1x:", int((df.groupby("content_id").size() > 1).sum()), "(expect 0)")

# 2) COUNTS: rows per client (unbalanced even inside the slice).
rpc = df.groupby("client_id").size()
print("rows/client: min", int(rpc.min()), "| median", int(rpc.median()), "| max", int(rpc.max()))

# 3) MISSINGNESS: overall, then by content_type to expose the PATTERN.
nulls = df.isna().mean().mul(100).round(1)
print("\ncolumns with missing values (% null):")
print(nulls[nulls > 0].sort_values(ascending=False).to_string())
pattern = pd.DataFrame({
    "word_count_%null":   df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean()*100, 1)),
    "keyword_data_%null": df.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean()*100, 1)),
})
print("\nmissingness by content_type (patterned, NOT random):")
print(pattern.to_string())

# 4) WINDOWS / sentinels: the label input is blank exactly when its denominator is 0.
print("\ntrend_pct blank:", int(df["trend_pct"].isna().sum()),
      "== impressions_prev_30d == 0:", int(df["impressions_prev_30d"].eq(0).sum()))
print("avg_position == 0 (no-data sentinel, not rank 0):", int(df["avg_position"].eq(0).sum()))

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this snapshot can never tell me:**

- **No forward outcome.** Features and the decline label share the *same* 90-day window — no calendar date, no future period — so I can only *describe* in-window decline, never honestly *predict* it. A true forward-looking label needs the Week-3 warehouse (`fact_content_daily_performance`, per `report_date`).
- **The label is a rule, not the world.** `is_declining_label` is defined from our own traffic (base rate ≈ 54.2%), so results stay **decision-support / directional** — never causal, never 'predicting Google'.
- **Sentinels & odd rates.** `avg_position == 0` means *no position data* (~4%), not rank 0; `scroll_rate` and `ai_traffic_pct` legitimately exceed 100 (independent numerator / denominator) — don't clip or impute them naively.
- **Warehouse limits I inherit at Week 3+** (invisible in this single snapshot): unbalanced per-client history (check `dim_clients.gsc_data_start`), GSC-only early rows zero-filled with `ga4_data_available = FALSE`, and the query table's 90-day window overlapping the label month (only `*_prev30` columns stay safe features).

In [ ]:
# (a) No forward window: label shares the SAME 90d window as the features.
print("report_date column present:", any("date" in c.lower() for c in df.columns))
print("=> decline is measured IN-window; a true forward label needs the warehouse (Week 3+).")

# (b) The label is a defined RULE, not an observed outcome.
print("label base rate (trend_direction == 'down'):",
      round(df["trend_direction"].eq("down").mean() * 100, 1), "%")

# (c) Rates can exceed 100 (different numerator/denominator systems) -> don't clip blindly.
print("scroll_rate > 100:", int((df["scroll_rate"] > 100).sum()),
      "| ai_traffic_pct > 100:", int((df["ai_traffic_pct"] > 100).sum()))

# (d) This slice is a single 90d window, so per-client history depth / GA4-availability /
#     query-window overlap only become checkable on the warehouse (Week 3+).
print("history depth here = one 90d snapshot -> gsc_data_start / ga4_data_available are warehouse-only checks.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.